# Aim
The aim of this notebook is to prototype a function to pair bb and cilia.

In [16]:
import pandas as pd

df = pd.read_excel(r'C:\Users\qfavey\Documents\Thomas-test-data\Test_Run\csv\all_cilia_features.xlsx')
df.head()

,filename,cilia_id,coords,distance_to_neurite_um,ratio,log_ratio,dt_neurite,dt_nuclei,log_dt_neurite,log_dt_nuclei,object_type,file_short,class
0,EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647...,1,"[np.float64(12.655128205128205), np.float64(43...",1.222115,6.726812,1.906101,0.304000,2.044951,-1.190728,0.715374,basal_body,S1,ambiguous
1,EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647...,2,"[np.float64(10.15625), np.float64(270.7109375)...",0.000000,5.257915,1.659734,0.760000,3.996015,-0.274437,1.385298,basal_body,S1,ambiguous
2,EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647...,3,"[np.float64(9.859866539561487), np.float64(520...",0.033143,9.844726,2.286936,0.429921,4.232454,-0.844154,1.442782,basal_body,S1,ambiguous
3,EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647...,4,"[np.float64(13.134453781512605), np.float64(44...",0.405344,6.726812,1.906101,0.304000,2.044951,-1.190728,0.715374,basal_body,S1,ambiguous
4,EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647...,5,"[np.float64(10.923404255319149), np.float64(13...",0.000000,0.174561,-1.745480,3.894135,0.679765,1.359472,-0.386009,basal_body,S1,soma


In [37]:
import re
import numpy as np

def parse_coords_string(series):
    """
    Convert a pandas series of strings like
    '[np.float64(10.52), np.float64(428.1), np.float64(11.57)]'
    into a proper numeric array (n_objects, 3)
    """
    coords_list = []
    float_pattern = re.compile(r"np\.float64\((.*?)\)")
    for s in series:
        # Find all numbers inside np.float64()
        numbers = float_pattern.findall(s)
        coords_list.append([float(x) for x in numbers])
    return np.array(coords_list, dtype=np.float64)

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

def pair_from_mixed_df(df):
    """
    Pair cilia and basal bodies from a single dataframe containing both.

    Assumes `coords` column is a numeric array (shape (3,)) for each row.

    Returns
    -------
    pd.DataFrame
        Original dataframe with:
        - paired_id
        - pairing_status ("paired" / "lonely")
        - pair_distance_um
    """
    df = df.copy()

    # -------------------------
    # Split objects
    # -------------------------
    df_cilia = df[df["object_type"] == "cilia"].copy()
    df_basal = df[df["object_type"] == "basal_body"].copy()

    # Initialize outputs
    df["paired_id"] = np.nan
    df["pair_distance_um"] = np.nan
    df["pairing_status"] = "lonely"

    if len(df_cilia) == 0 or len(df_basal) == 0:
        # No possible pairs
        return df

    # -------------------------
    # Build coordinate arrays
    # -------------------------
    
    cilia_coords = parse_coords_string(df_cilia["coords"])  # shape (n_cilia, 3)
    basal_coords = parse_coords_string(df_basal["coords"])  # shape (n_basal, 3)

    # -------------------------
    # Distance matrix
    # -------------------------
    dist_matrix = np.linalg.norm(
        cilia_coords[:, None, :] - basal_coords[None, :, :],
        axis=2
    )
    print(dist_matrix)
    dist_mask = dist_matrix.copy()


    # -------------------------
    # Hungarian assignment
    # -------------------------
    c_idx, b_idx = linear_sum_assignment(dist_mask)

    # -------------------------
    # Assign pairs
    # -------------------------
    for c, b in zip(c_idx, b_idx):
        if dist_mask[c, b] != np.inf:
            cilia_row_idx = df_cilia.index[c]
            basal_row_idx = df_basal.index[b]

            dist = dist_mask[c, b]
            cilia_id = df_cilia.loc[cilia_row_idx, "cilia_id"]
            basal_id = df_basal.loc[basal_row_idx, "cilia_id"]

            # Assign to both
            df.loc[cilia_row_idx, ["paired_id", "pair_distance_um", "pairing_status"]] = [
                basal_id, dist, "paired"
            ]
            df.loc[basal_row_idx, ["paired_id", "pair_distance_um", "pairing_status"]] = [
                cilia_id, dist, "paired"
            ]

    return df

In [56]:
df2 = pair_from_mixed_df(df)

UFuncTypeError: ufunc 'subtract' did not contain a loop with signature matching types (dtype('<U96'), dtype('<U96')) -> None

In [54]:
df2

,filename,cilia_id,coords,distance_to_neurite_um,ratio,log_ratio,dt_neurite,dt_nuclei,log_dt_neurite,log_dt_nuclei,object_type,file_short,class,paired_id,pair_distance_um,pairing_status
0,EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647...,1,"[np.float64(12.655128205128205), np.float64(43...",1.222115,6.726812,1.906101,0.304000,2.044951,-1.190728,0.715374,basal_body,S1,ambiguous,1.0,4.677405,paired
1,EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647...,2,"[np.float64(10.15625), np.float64(270.7109375)...",0.000000,5.257915,1.659734,0.760000,3.996015,-0.274437,1.385298,basal_body,S1,ambiguous,3.0,4.491678,paired
2,EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647...,3,"[np.float64(9.859866539561487), np.float64(520...",0.033143,9.844726,2.286936,0.429921,4.232454,-0.844154,1.442782,basal_body,S1,ambiguous,2.0,11.102538,paired
3,EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647...,4,"[np.float64(13.134453781512605), np.float64(44...",0.405344,6.726812,1.906101,0.304000,2.044951,-1.190728,0.715374,basal_body,S1,ambiguous,NaN,NaN,lonely
4,EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647...,5,"[np.float64(10.923404255319149), np.float64(13...",0.000000,0.174561,-1.745480,3.894135,0.679765,1.359472,-0.386009,basal_body,S1,soma,NaN,NaN,lonely
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
500,EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647...,109,"[np.float64(8.39516129032258), np.float64(1418...",1.680368,7.385016,1.999453,0.548044,4.047312,-0.601400,1.398053,cilia,S3,ambiguous,168.0,8.617768,paired
501,EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647...,110,"[np.float64(12.696969696969697), np.float64(61...",0.457203,9.219544,2.221326,0.152000,1.401371,-1.883875,0.337451,cilia,S3,ambiguous,177.0,101.895071,paired
502,EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647...,111,"[np.float64(8.595588235294118), np.float64(785...",0.383819,5.385165,1.683648,0.760000,4.092725,-0.274437,1.409211,cilia,S3,ambiguous,171.0,62.132523,paired
503,EXP_E6_d40_WTSII_DAPI_PCN488_TUJ1568_ARL13B647...,112,"[np.float64(8.901234567901234), np.float64(106...",0.000000,0.000000,-inf,1.452766,0.000000,0.373469,-inf,cilia,S3,soma,169.0,7.343948,paired
